# 当前论文表3的MobileNetV3消融

当前论文表3的MobileNetV3消融。依赖主实验已保存的Full检查点；先15后16。原cell5的Full结果不在此处重跑。

来源为作者提供的APTOS_retest.ipynb，2026年10月15日按用途筛选。
代码与保存的输出逐格原样复制；不是重新运行的结果。原文件另行完整保留。
原来的训练开关、路径和依赖安装命令均保留，运行前需要配置对应数据与环境。
下面的编号是原文件从1开始的单元格编号。


## 原单元格 2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 原单元格 3


In [ ]:
import zipfile
import os

# 1. 定义压缩包在云盘里的位置
# 假设你直接传到了根目录，那就是这个路径：
zip_path = '/content/drive/MyDrive/split_data_preprocessed.zip'

# 2. 定义你想解压到哪里
# 建议解压到 Colab 的临时空间 (/content)，读取速度比直接读云盘快 10 倍！
extract_path = '/content'

# 3. 开始解压
if os.path.exists(extract_path):
  print("开始解压... (可能需要几分钟)")
  with zipfile.ZipFile(zip_path, 'r') as zip_ref:
      zip_ref.extractall(extract_path)
  print("解压完成！")


## 原单元格 15


In [ ]:
# ============================================================
# MobileNetV3-Large fixed-split strategy check
# Train only missing variants:
#   B: CE + Augmentation
#   C: CE + Augmentation + Weighted Sampler
#
# Do NOT retrain Full.
# Evaluate existing Full checkpoint:
#   D: Focal + Augmentation + Weighted Sampler
# ============================================================

!pip -q install timm openpyxl

import re
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    roc_auc_score
)

from google.colab import drive
drive.mount("/content/drive")


# ============================================================
# Config
# ============================================================

SEED = 42
NUM_CLASSES = 5
IMG_SIZE = 224

BATCH_SIZE = 32
EPOCHS = 40
LR = 3e-4
PATIENCE = 10
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SPLIT_ROOT = Path("/content/split_data_preprocessed")
TRAIN_DIR = SPLIT_ROOT / "train"
VAL_DIR = SPLIT_ROOT / "val"

MODEL_LABEL = "MobileNetV3-Large"
TIMM_NAME = "mobilenetv3_large_100"

OUT_DIR = Path("/content/drive/MyDrive/dr_fixed_split_strategy_check_mobilenetv3_224_missing_only")
CKPT_DIR = OUT_DIR / "checkpoints"
TABLE_DIR = OUT_DIR / "tables"
PRED_DIR = OUT_DIR / "predictions"

for d in [OUT_DIR, CKPT_DIR, TABLE_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 你已经跑过的 Full checkpoint，不重训，只评估
EXISTING_FULL_CKPT = Path(
    "/content/drive/MyDrive/dr_fixed_split_results/checkpoints/best_fixedsplit_mobilenetv3_large.pth"
)

if not EXISTING_FULL_CKPT.exists():
    raise FileNotFoundError(f"找不到已有 Full checkpoint: {EXISTING_FULL_CKPT}")


TRAIN_VARIANTS = [
    {
        "variant_id": "B_CE_noSampler_Aug",
        "variant_name": "CE + Augmentation",
        "loss": "ce",
        "use_sampler": False,
        "ckpt_path": CKPT_DIR / "mobilenetv3_B_CE_noSampler_Aug.pth"
    },
    {
        "variant_id": "C_CE_sampler_Aug",
        "variant_name": "CE + Augmentation + Weighted Sampler",
        "loss": "ce",
        "use_sampler": True,
        "ckpt_path": CKPT_DIR / "mobilenetv3_C_CE_sampler_Aug.pth"
    }
]

EVAL_EXISTING_FULL = {
    "variant_id": "D_existing_Focal_sampler_Aug",
    "variant_name": "Existing Full: Focal + Augmentation + Weighted Sampler",
    "loss": "focal",
    "use_sampler": True,
    "ckpt_path": EXISTING_FULL_CKPT
}


# ============================================================
# Seed
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)


# ============================================================
# Dataset
# ============================================================

class FixedFolderDRDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        class_dirs = sorted([d for d in self.root_dir.iterdir() if d.is_dir()], key=lambda x: x.name)

        for class_dir in class_dirs:
            m = re.match(r"^(\d+)", class_dir.name)
            if not m:
                raise ValueError(f"类别文件夹名必须以 0-4 开头: {class_dir.name}")

            label = int(m.group(1))

            files = []
            for ext in ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"]:
                files.extend(class_dir.glob(ext))
                files.extend(class_dir.glob(ext.upper()))

            for img_path in sorted(files, key=lambda x: x.name):
                self.samples.append((str(img_path), label))

        if len(self.samples) == 0:
            raise ValueError(f"No images found in {root_dir}")

        self.df = pd.DataFrame(self.samples, columns=["path", "label"])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        image_id = Path(path).name
        return image, label, path, image_id


# ============================================================
# Transforms
# ============================================================

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


# ============================================================
# Loss
# ============================================================

def get_criterion(loss_name):
    if loss_name == "ce":
        return nn.CrossEntropyLoss()
    else:
        raise ValueError("This script only trains CE variants. Full Focal checkpoint is only evaluated.")


# ============================================================
# Metrics
# ============================================================

def compute_metrics(y_true, y_pred, probs):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        output_dict=True,
        zero_division=0
    )

    per_class_rows = []
    for c in range(NUM_CLASSES):
        TP = cm[c, c]
        FP = cm[:, c].sum() - TP
        FN = cm[c, :].sum() - TP
        TN = cm.sum() - TP - FP - FN

        specificity = TN / (TN + FP) if (TN + FP) > 0 else np.nan

        per_class_rows.append({
            "class": c,
            "precision": report[str(c)]["precision"],
            "recall": report[str(c)]["recall"],
            "f1": report[str(c)]["f1-score"],
            "specificity": specificity,
            "support": report[str(c)]["support"]
        })

    ref_true = np.isin(y_true, [2, 3, 4]).astype(int)
    ref_pred = np.isin(y_pred, [2, 3, 4]).astype(int)
    ref_score = probs[:, 2] + probs[:, 3] + probs[:, 4]

    tn, fp, fn, tp = confusion_matrix(ref_true, ref_pred, labels=[0, 1]).ravel()

    ref_sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    ref_spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ref_precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    ref_f1 = 2 * ref_precision * ref_sens / (ref_precision + ref_sens) if (ref_precision + ref_sens) > 0 else np.nan
    ref_acc = (tp + tn) / (tp + tn + fp + fn)

    try:
        ref_auc = roc_auc_score(ref_true, ref_score)
    except Exception:
        ref_auc = np.nan

    summary = {
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "qwk": qwk,
        "referable_sensitivity": ref_sens,
        "referable_specificity": ref_spec,
        "referable_precision": ref_precision,
        "referable_f1": ref_f1,
        "referable_accuracy": ref_acc,
        "referable_auc": ref_auc,
        "referable_tp": tp,
        "referable_tn": tn,
        "referable_fp": fp,
        "referable_fn": fn
    }

    return summary, pd.DataFrame(per_class_rows), cm


# ============================================================
# Dataloaders
# ============================================================

def build_loaders(use_sampler):
    train_dataset = FixedFolderDRDataset(TRAIN_DIR, transform=train_transforms)
    val_dataset = FixedFolderDRDataset(VAL_DIR, transform=val_transforms)

    print("Train size:", len(train_dataset))
    print("Val size:", len(val_dataset))
    print("Train counts:")
    print(train_dataset.df["label"].value_counts().sort_index())
    print("Val counts:")
    print(val_dataset.df["label"].value_counts().sort_index())

    if use_sampler:
        labels = train_dataset.df["label"].values
        class_counts = np.bincount(labels, minlength=NUM_CLASSES)
        class_weights = 1.0 / np.maximum(class_counts, 1)
        sample_weights = class_weights[labels]

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=NUM_WORKERS,
            pin_memory=True
        )
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_loader


def build_val_loader():
    val_dataset = FixedFolderDRDataset(VAL_DIR, transform=val_transforms)
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    return val_loader


# ============================================================
# Evaluation
# ============================================================

def evaluate_model(model, loader, variant_id, variant_name, save_predictions=False):
    model.eval()

    rows = []
    y_true_all = []
    y_pred_all = []
    probs_all = []

    with torch.no_grad():
        for images, labels, paths, image_ids in tqdm(loader, desc=f"Eval {variant_id}", leave=False):
            images = images.to(DEVICE)

            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)

            labels_np = labels.numpy()

            y_true_all.extend(labels_np.tolist())
            y_pred_all.extend(preds.tolist())
            probs_all.append(probs)

            if save_predictions:
                for i in range(len(labels_np)):
                    rows.append({
                        "model": MODEL_LABEL,
                        "variant_id": variant_id,
                        "variant_name": variant_name,
                        "image_id": image_ids[i],
                        "image_path": paths[i],
                        "y_true": int(labels_np[i]),
                        "y_pred": int(preds[i]),
                        "prob_0": float(probs[i, 0]),
                        "prob_1": float(probs[i, 1]),
                        "prob_2": float(probs[i, 2]),
                        "prob_3": float(probs[i, 3]),
                        "prob_4": float(probs[i, 4]),
                        "referable_score": float(probs[i, 2] + probs[i, 3] + probs[i, 4])
                    })

    probs_all = np.vstack(probs_all)
    summary, per_class_df, cm = compute_metrics(y_true_all, y_pred_all, probs_all)

    pred_df = pd.DataFrame(rows) if save_predictions else None
    return summary, per_class_df, cm, pred_df


# ============================================================
# Train one CE variant
# ============================================================

def train_one_variant(cfg):
    set_seed(SEED)

    variant_id = cfg["variant_id"]
    variant_name = cfg["variant_name"]

    print("\n" + "=" * 90)
    print(f"Training: {MODEL_LABEL} | {variant_name}")
    print("=" * 90)

    train_loader, val_loader = build_loaders(use_sampler=cfg["use_sampler"])

    model = timm.create_model(TIMM_NAME, pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(DEVICE)

    criterion = get_criterion(cfg["loss"])
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_qwk = -999
    best_epoch = -1
    patience_count = 0
    history_rows = []

    for epoch in range(EPOCHS):
        model.train()
        losses = []

        for images, labels, _, _ in tqdm(train_loader, desc=f"{variant_id} epoch {epoch+1}/{EPOCHS}", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            losses.append(loss.item())

        val_summary, _, _, _ = evaluate_model(
            model,
            val_loader,
            variant_id=variant_id,
            variant_name=variant_name,
            save_predictions=False
        )

        history_rows.append({
            "model": MODEL_LABEL,
            "variant_id": variant_id,
            "variant_name": variant_name,
            "epoch": epoch + 1,
            "train_loss": float(np.mean(losses)),
            "val_accuracy": val_summary["accuracy"],
            "val_macro_f1": val_summary["macro_f1"],
            "val_qwk": val_summary["qwk"]
        })

        print(
            f"Epoch {epoch+1}: "
            f"loss={np.mean(losses):.4f}, "
            f"acc={val_summary['accuracy']*100:.2f}%, "
            f"macro_f1={val_summary['macro_f1']*100:.2f}%, "
            f"qwk={val_summary['qwk']:.4f}"
        )

        if val_summary["qwk"] > best_qwk:
            best_qwk = val_summary["qwk"]
            best_epoch = epoch + 1
            torch.save(model.state_dict(), cfg["ckpt_path"])
            print(f"  Saved best checkpoint. Best QWK={best_qwk:.4f}")
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print("  Early stopping triggered.")
                break

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(TABLE_DIR / f"history_{variant_id}.csv", index=False)

    print(f"Best epoch: {best_epoch}, Best QWK: {best_qwk:.4f}")
    return history_df


# ============================================================
# Run only missing CE variants
# ============================================================

all_history = []

for cfg in TRAIN_VARIANTS:
    history_df = train_one_variant(cfg)
    all_history.append(history_df)


# ============================================================
# Final evaluation:
# B, C from newly trained checkpoints
# D from existing Full checkpoint
# ============================================================

EVAL_VARIANTS = TRAIN_VARIANTS + [EVAL_EXISTING_FULL]

summary_rows = []
per_class_rows = []
confusion_rows = []
prediction_dfs = []

val_loader = build_val_loader()

for cfg in EVAL_VARIANTS:
    variant_id = cfg["variant_id"]
    variant_name = cfg["variant_name"]

    print("\n" + "=" * 90)
    print(f"Final evaluation: {variant_name}")
    print(f"Checkpoint: {cfg['ckpt_path']}")
    print("=" * 90)

    model = timm.create_model(TIMM_NAME, pretrained=False, num_classes=NUM_CLASSES)
    state = torch.load(cfg["ckpt_path"], map_location=DEVICE)

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]

    clean_state = {}
    for k, v in state.items():
        clean_state[k.replace("module.", "")] = v

    model.load_state_dict(clean_state, strict=True)
    model = model.to(DEVICE)

    summary, pc_df, cm, pred_df = evaluate_model(
        model,
        val_loader,
        variant_id=variant_id,
        variant_name=variant_name,
        save_predictions=True
    )

    summary_row = {
        "model": MODEL_LABEL,
        "timm_name": TIMM_NAME,
        "variant_id": variant_id,
        "variant_name": variant_name,
        "loss": cfg["loss"],
        "use_sampler": cfg["use_sampler"],
        "use_augmentation": True,
        "checkpoint_path": str(cfg["ckpt_path"])
    }
    summary_row.update(summary)

    for _, r in pc_df.iterrows():
        c = int(r["class"])
        summary_row[f"class{c}_precision"] = r["precision"]
        summary_row[f"class{c}_recall"] = r["recall"]
        summary_row[f"class{c}_f1"] = r["f1"]
        summary_row[f"class{c}_specificity"] = r["specificity"]
        summary_row[f"class{c}_support"] = r["support"]

    summary_rows.append(summary_row)

    pc_df.insert(0, "model", MODEL_LABEL)
    pc_df.insert(1, "variant_id", variant_id)
    pc_df.insert(2, "variant_name", variant_name)
    per_class_rows.append(pc_df)

    for t in range(NUM_CLASSES):
        for p in range(NUM_CLASSES):
            confusion_rows.append({
                "model": MODEL_LABEL,
                "variant_id": variant_id,
                "variant_name": variant_name,
                "true_class": t,
                "pred_class": p,
                "count": int(cm[t, p])
            })

    prediction_dfs.append(pred_df)


summary_df = pd.DataFrame(summary_rows)
per_class_df = pd.concat(per_class_rows, ignore_index=True)
confusion_df = pd.DataFrame(confusion_rows)
history_df = pd.concat(all_history, ignore_index=True)
predictions_df = pd.concat(prediction_dfs, ignore_index=True)


# ============================================================
# Save
# ============================================================

summary_path = TABLE_DIR / "mobilenetv3_missing_strategy_summary_with_existing_full.csv"
per_class_path = TABLE_DIR / "mobilenetv3_missing_strategy_per_class_with_existing_full.csv"
confusion_path = TABLE_DIR / "mobilenetv3_missing_strategy_confusion_with_existing_full.csv"
history_path = TABLE_DIR / "mobilenetv3_missing_strategy_history.csv"
predictions_path = PRED_DIR / "mobilenetv3_missing_strategy_predictions_with_existing_full.csv"
excel_path = TABLE_DIR / "mobilenetv3_missing_strategy_all_results_with_existing_full.xlsx"

summary_df.to_csv(summary_path, index=False)
per_class_df.to_csv(per_class_path, index=False)
confusion_df.to_csv(confusion_path, index=False)
history_df.to_csv(history_path, index=False)
predictions_df.to_csv(predictions_path, index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    per_class_df.to_excel(writer, sheet_name="per_class", index=False)
    confusion_df.to_excel(writer, sheet_name="confusion", index=False)
    history_df.to_excel(writer, sheet_name="history", index=False)
    predictions_df.to_excel(writer, sheet_name="predictions", index=False)

print("\nFinished.")
print("Summary:", summary_path)
print("Per-class:", per_class_path)
print("Predictions:", predictions_path)
print("Excel:", excel_path)

display(summary_df)

## 原单元格 16


In [ ]:
# ============================================================
# MobileNetV3-Large old-order ablation completion
# Train only:
# 1. Baseline: CE + no sampler + no augmentation
# 2. + Sampler: CE + Weighted Sampler + no augmentation
#
# Evaluate existing:
# 3. + Sampler + Augmentation: CE + Weighted Sampler + Augmentation
# 4. Proposed: Focal + Weighted Sampler + Augmentation
# ============================================================

!pip -q install timm openpyxl

import re, random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    roc_auc_score
)

from google.colab import drive
drive.mount("/content/drive")


# ============================================================
# Config
# ============================================================

SEED = 42
NUM_CLASSES = 5
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 40
LR = 3e-4
PATIENCE = 10
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SPLIT_ROOT = Path("/content/split_data_preprocessed")
TRAIN_DIR = SPLIT_ROOT / "train"
VAL_DIR = SPLIT_ROOT / "val"

MODEL_LABEL = "MobileNetV3-Large"
TIMM_NAME = "mobilenetv3_large_100"

OUT_DIR = Path("/content/drive/MyDrive/dr_fixed_split_ablation_mobilenetv3_old_order_224")
CKPT_DIR = OUT_DIR / "checkpoints"
TABLE_DIR = OUT_DIR / "tables"
PRED_DIR = OUT_DIR / "predictions"

for d in [OUT_DIR, CKPT_DIR, TABLE_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 你之前已经训练好的 checkpoint
EXISTING_CE_SAMPLER_AUG_CKPT = Path(
    "/content/drive/MyDrive/dr_fixed_split_strategy_check_mobilenetv3_224_missing_only/checkpoints/mobilenetv3_C_CE_sampler_Aug.pth"
)

EXISTING_PROPOSED_CKPT = Path(
    "/content/drive/MyDrive/dr_fixed_split_results/checkpoints/best_fixedsplit_mobilenetv3_large.pth"
)

if not EXISTING_CE_SAMPLER_AUG_CKPT.exists():
    print("WARNING: 找不到已有 CE + Sampler + Aug checkpoint:", EXISTING_CE_SAMPLER_AUG_CKPT)

if not EXISTING_PROPOSED_CKPT.exists():
    raise FileNotFoundError(f"找不到已有 Proposed checkpoint: {EXISTING_PROPOSED_CKPT}")


TRAIN_VARIANTS = [
    {
        "variant_id": "A_baseline_CE_noSampler_noAug",
        "variant_name": "Baseline",
        "loss": "ce",
        "use_sampler": False,
        "use_augmentation": False,
        "ckpt_path": CKPT_DIR / "mobilenetv3_A_baseline_CE_noSampler_noAug.pth"
    },
    {
        "variant_id": "B_CE_sampler_noAug",
        "variant_name": "+ Sampler",
        "loss": "ce",
        "use_sampler": True,
        "use_augmentation": False,
        "ckpt_path": CKPT_DIR / "mobilenetv3_B_CE_sampler_noAug.pth"
    }
]

EVAL_VARIANTS = TRAIN_VARIANTS.copy()

if EXISTING_CE_SAMPLER_AUG_CKPT.exists():
    EVAL_VARIANTS.append({
        "variant_id": "C_CE_sampler_Aug_existing",
        "variant_name": "+ Sampler + Augmentation",
        "loss": "ce",
        "use_sampler": True,
        "use_augmentation": True,
        "ckpt_path": EXISTING_CE_SAMPLER_AUG_CKPT
    })

EVAL_VARIANTS.append({
    "variant_id": "D_Proposed_focal_sampler_Aug_existing",
    "variant_name": "Proposed",
    "loss": "focal",
    "use_sampler": True,
    "use_augmentation": True,
    "ckpt_path": EXISTING_PROPOSED_CKPT
})


# ============================================================
# Seed
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)


# ============================================================
# Dataset
# ============================================================

class FixedFolderDRDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        class_dirs = sorted([d for d in self.root_dir.iterdir() if d.is_dir()], key=lambda x: x.name)

        for class_dir in class_dirs:
            m = re.match(r"^(\d+)", class_dir.name)
            if not m:
                raise ValueError(f"Class folder must start with 0-4: {class_dir.name}")

            label = int(m.group(1))

            files = []
            for ext in ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"]:
                files.extend(class_dir.glob(ext))
                files.extend(class_dir.glob(ext.upper()))

            for img_path in sorted(files, key=lambda x: x.name):
                self.samples.append((str(img_path), label))

        self.df = pd.DataFrame(self.samples, columns=["path", "label"])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label, path, Path(path).name


# ============================================================
# Transforms
# ============================================================

basic_train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

aug_train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


# ============================================================
# Loss
# ============================================================

def get_criterion(loss_name):
    if loss_name == "ce":
        return nn.CrossEntropyLoss()
    raise ValueError("This script only trains CE variants.")


# ============================================================
# Metrics
# ============================================================

def compute_metrics(y_true, y_pred, probs):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        output_dict=True,
        zero_division=0
    )

    per_class_rows = []
    for c in range(NUM_CLASSES):
        TP = cm[c, c]
        FP = cm[:, c].sum() - TP
        FN = cm[c, :].sum() - TP
        TN = cm.sum() - TP - FP - FN
        specificity = TN / (TN + FP) if (TN + FP) > 0 else np.nan

        per_class_rows.append({
            "class": c,
            "precision": report[str(c)]["precision"],
            "recall": report[str(c)]["recall"],
            "f1": report[str(c)]["f1-score"],
            "specificity": specificity,
            "support": report[str(c)]["support"]
        })

    ref_true = np.isin(y_true, [2, 3, 4]).astype(int)
    ref_pred = np.isin(y_pred, [2, 3, 4]).astype(int)
    ref_score = probs[:, 2] + probs[:, 3] + probs[:, 4]

    tn, fp, fn, tp = confusion_matrix(ref_true, ref_pred, labels=[0, 1]).ravel()

    ref_sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    ref_spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ref_precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    ref_f1 = 2 * ref_precision * ref_sens / (ref_precision + ref_sens) if (ref_precision + ref_sens) > 0 else np.nan
    ref_acc = (tp + tn) / (tp + tn + fp + fn)

    try:
        ref_auc = roc_auc_score(ref_true, ref_score)
    except Exception:
        ref_auc = np.nan

    summary = {
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "qwk": qwk,
        "referable_sensitivity": ref_sens,
        "referable_specificity": ref_spec,
        "referable_precision": ref_precision,
        "referable_f1": ref_f1,
        "referable_accuracy": ref_acc,
        "referable_auc": ref_auc,
        "referable_tp": tp,
        "referable_tn": tn,
        "referable_fp": fp,
        "referable_fn": fn
    }

    return summary, pd.DataFrame(per_class_rows), cm


# ============================================================
# Loaders
# ============================================================

def build_loaders(use_sampler, use_augmentation):
    train_transform = aug_train_transforms if use_augmentation else basic_train_transforms

    train_dataset = FixedFolderDRDataset(TRAIN_DIR, transform=train_transform)
    val_dataset = FixedFolderDRDataset(VAL_DIR, transform=val_transforms)

    if use_sampler:
        labels = train_dataset.df["label"].values
        class_counts = np.bincount(labels, minlength=NUM_CLASSES)
        class_weights = 1.0 / np.maximum(class_counts, 1)
        sample_weights = class_weights[labels]

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=NUM_WORKERS,
            pin_memory=True
        )
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_loader


def build_val_loader():
    val_dataset = FixedFolderDRDataset(VAL_DIR, transform=val_transforms)
    return DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )


# ============================================================
# Evaluation
# ============================================================

def load_state_dict_clean(model, ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE)

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]

    clean_state = {}
    for k, v in state.items():
        clean_state[k.replace("module.", "")] = v

    model.load_state_dict(clean_state, strict=True)
    return model


def evaluate_model(model, loader, variant_id, variant_name, save_predictions=False):
    model.eval()

    rows = []
    y_true_all, y_pred_all, probs_all = [], [], []

    with torch.no_grad():
        for images, labels, paths, image_ids in tqdm(loader, desc=f"Eval {variant_id}", leave=False):
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            labels_np = labels.numpy()

            y_true_all.extend(labels_np.tolist())
            y_pred_all.extend(preds.tolist())
            probs_all.append(probs)

            if save_predictions:
                for i in range(len(labels_np)):
                    rows.append({
                        "model": MODEL_LABEL,
                        "variant_id": variant_id,
                        "variant_name": variant_name,
                        "image_id": image_ids[i],
                        "image_path": paths[i],
                        "y_true": int(labels_np[i]),
                        "y_pred": int(preds[i]),
                        "prob_0": float(probs[i, 0]),
                        "prob_1": float(probs[i, 1]),
                        "prob_2": float(probs[i, 2]),
                        "prob_3": float(probs[i, 3]),
                        "prob_4": float(probs[i, 4]),
                        "referable_score": float(probs[i, 2] + probs[i, 3] + probs[i, 4])
                    })

    probs_all = np.vstack(probs_all)
    summary, per_class_df, cm = compute_metrics(y_true_all, y_pred_all, probs_all)

    pred_df = pd.DataFrame(rows) if save_predictions else None
    return summary, per_class_df, cm, pred_df


# ============================================================
# Train missing variants
# ============================================================

def train_one_variant(cfg):
    set_seed(SEED)

    print("\n" + "=" * 90)
    print(f"Training {cfg['variant_name']}")
    print("=" * 90)

    train_loader, val_loader = build_loaders(
        use_sampler=cfg["use_sampler"],
        use_augmentation=cfg["use_augmentation"]
    )

    model = timm.create_model(TIMM_NAME, pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(DEVICE)

    criterion = get_criterion(cfg["loss"])
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_qwk = -999
    best_epoch = -1
    patience_count = 0
    history_rows = []

    for epoch in range(EPOCHS):
        model.train()
        losses = []

        for images, labels, _, _ in tqdm(train_loader, desc=f"{cfg['variant_id']} epoch {epoch+1}", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            losses.append(loss.item())

        val_summary, _, _, _ = evaluate_model(
            model,
            val_loader,
            variant_id=cfg["variant_id"],
            variant_name=cfg["variant_name"],
            save_predictions=False
        )

        history_rows.append({
            "model": MODEL_LABEL,
            "variant_id": cfg["variant_id"],
            "variant_name": cfg["variant_name"],
            "epoch": epoch + 1,
            "train_loss": float(np.mean(losses)),
            "val_accuracy": val_summary["accuracy"],
            "val_macro_f1": val_summary["macro_f1"],
            "val_qwk": val_summary["qwk"]
        })

        print(
            f"Epoch {epoch+1}: "
            f"loss={np.mean(losses):.4f}, "
            f"acc={val_summary['accuracy']*100:.2f}%, "
            f"macro_f1={val_summary['macro_f1']*100:.2f}%, "
            f"qwk={val_summary['qwk']:.4f}"
        )

        if val_summary["qwk"] > best_qwk:
            best_qwk = val_summary["qwk"]
            best_epoch = epoch + 1
            torch.save(model.state_dict(), cfg["ckpt_path"])
            print(f"Saved best checkpoint. QWK={best_qwk:.4f}")
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print("Early stopping triggered.")
                break

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(TABLE_DIR / f"history_{cfg['variant_id']}.csv", index=False)

    print(f"Best epoch: {best_epoch}, best QWK: {best_qwk:.4f}")
    return history_df


all_history = []

for cfg in TRAIN_VARIANTS:
    history_df = train_one_variant(cfg)
    all_history.append(history_df)


# ============================================================
# Final evaluation all old-order variants
# ============================================================

val_loader = build_val_loader()

summary_rows = []
per_class_rows = []
confusion_rows = []
prediction_rows = []

for cfg in EVAL_VARIANTS:
    print("\n" + "=" * 90)
    print(f"Final evaluation: {cfg['variant_name']}")
    print(f"Checkpoint: {cfg['ckpt_path']}")
    print("=" * 90)

    model = timm.create_model(TIMM_NAME, pretrained=False, num_classes=NUM_CLASSES)
    model = load_state_dict_clean(model, cfg["ckpt_path"])
    model = model.to(DEVICE)

    summary, pc_df, cm, pred_df = evaluate_model(
        model,
        val_loader,
        variant_id=cfg["variant_id"],
        variant_name=cfg["variant_name"],
        save_predictions=True
    )

    summary_row = {
        "model": MODEL_LABEL,
        "timm_name": TIMM_NAME,
        "variant_id": cfg["variant_id"],
        "variant_name": cfg["variant_name"],
        "loss": cfg["loss"],
        "use_sampler": cfg["use_sampler"],
        "use_augmentation": cfg["use_augmentation"],
        "checkpoint_path": str(cfg["ckpt_path"])
    }
    summary_row.update(summary)

    for _, r in pc_df.iterrows():
        c = int(r["class"])
        summary_row[f"class{c}_precision"] = r["precision"]
        summary_row[f"class{c}_recall"] = r["recall"]
        summary_row[f"class{c}_f1"] = r["f1"]
        summary_row[f"class{c}_specificity"] = r["specificity"]
        summary_row[f"class{c}_support"] = r["support"]

    summary_rows.append(summary_row)

    pc_df.insert(0, "model", MODEL_LABEL)
    pc_df.insert(1, "variant_id", cfg["variant_id"])
    pc_df.insert(2, "variant_name", cfg["variant_name"])
    per_class_rows.append(pc_df)

    for t in range(NUM_CLASSES):
        for p in range(NUM_CLASSES):
            confusion_rows.append({
                "model": MODEL_LABEL,
                "variant_id": cfg["variant_id"],
                "variant_name": cfg["variant_name"],
                "true_class": t,
                "pred_class": p,
                "count": int(cm[t, p])
            })

    prediction_rows.append(pred_df)


summary_df = pd.DataFrame(summary_rows)
per_class_df = pd.concat(per_class_rows, ignore_index=True)
confusion_df = pd.DataFrame(confusion_rows)
predictions_df = pd.concat(prediction_rows, ignore_index=True)
history_df = pd.concat(all_history, ignore_index=True)


summary_path = TABLE_DIR / "mobilenetv3_old_order_ablation_summary.csv"
per_class_path = TABLE_DIR / "mobilenetv3_old_order_ablation_per_class.csv"
confusion_path = TABLE_DIR / "mobilenetv3_old_order_ablation_confusion.csv"
predictions_path = PRED_DIR / "mobilenetv3_old_order_ablation_predictions.csv"
history_path = TABLE_DIR / "mobilenetv3_old_order_ablation_history.csv"
excel_path = TABLE_DIR / "mobilenetv3_old_order_ablation_all_results.xlsx"

summary_df.to_csv(summary_path, index=False)
per_class_df.to_csv(per_class_path, index=False)
confusion_df.to_csv(confusion_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
history_df.to_csv(history_path, index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    per_class_df.to_excel(writer, sheet_name="per_class", index=False)
    confusion_df.to_excel(writer, sheet_name="confusion", index=False)
    predictions_df.to_excel(writer, sheet_name="predictions", index=False)
    history_df.to_excel(writer, sheet_name="history", index=False)

print("\nFinished.")
print("Summary:", summary_path)
print("Per-class:", per_class_path)
print("Predictions:", predictions_path)
print("Excel:", excel_path)

display(summary_df)